In [1]:
import pandas as pd 
import plotly.express as px

In [2]:
df = pd.read_csv('cases_deaths.csv')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 558258 entries, 0 to 558257
Data columns (total 40 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   country                                  558258 non-null  str    
 1   date                                     558258 non-null  str    
 2   new_cases                                554318 non-null  float64
 3   total_cases                              558258 non-null  int64  
 4   new_deaths                               555343 non-null  float64
 5   total_deaths                             558258 non-null  int64  
 6   weekly_cases                             553063 non-null  float64
 7   weekly_deaths                            554098 non-null  float64
 8   weekly_pct_growth_cases                  341333 non-null  float64
 9   weekly_pct_growth_deaths                 217133 non-null  float64
 10  biweekly_cases                           55

In [8]:
df.columns

Index(['country', 'date', 'new_cases', 'total_cases', 'new_deaths',
       'total_deaths', 'weekly_cases', 'weekly_deaths',
       'weekly_pct_growth_cases', 'weekly_pct_growth_deaths', 'biweekly_cases',
       'biweekly_deaths', 'biweekly_pct_growth_cases',
       'biweekly_pct_growth_deaths', 'new_cases_per_million',
       'new_deaths_per_million', 'total_cases_per_million',
       'total_deaths_per_million', 'weekly_cases_per_million',
       'weekly_deaths_per_million', 'biweekly_cases_per_million',
       'biweekly_deaths_per_million', 'total_deaths_per_100k',
       'new_deaths_per_100k', 'new_cases_7_day_avg_right',
       'new_deaths_7_day_avg_right', 'new_cases_per_million_7_day_avg_right',
       'new_deaths_per_million_7_day_avg_right',
       'new_deaths_per_100k_7_day_avg_right', 'cfr', 'cfr_100_cases',
       'cfr_short_term', 'days_since_100_total_cases',
       'days_since_5_total_deaths', 'days_since_1_total_cases_per_million',
       'days_since_0_1_total_deaths_per_

In [9]:
df.groupby('country')['total_cases_per_million'].sum()

country
Afghanistan                                            9.367515e+06
Africa                                                 1.551890e+07
Albania                                                1.937371e+08
Algeria                                                1.048061e+07
American Samoa                                         2.410524e+08
                                                           ...     
World excl. China and South Korea                      1.537997e+08
World excl. China, South Korea, Japan and Singapore    1.494568e+08
Yemen                                                  5.437436e+05
Zambia                                                 2.843207e+07
Zimbabwe                                               2.663042e+07
Name: total_cases_per_million, Length: 249, dtype: float64

In [3]:
regions = [
    # Continents / agrégats
    'Africa', 'Asia', 'Asia excl. China', 'Europe', 'European Union (27)',
    'High-income countries', 'Low-income countries', 'Lower-middle-income countries',
    'North America', 'Oceania', 'South America', 'Upper-middle-income countries',
    'World', 'World excl. China', 'World excl. China and South Korea',
    'World excl. China, South Korea, Japan and Singapore',
    # Territoires US
    'American Samoa', 'Guam', 'Northern Mariana Islands', 'Puerto Rico',
    'United States Virgin Islands',
    # Territoires UK
    'Anguilla', 'Bermuda', 'British Virgin Islands', 'Cayman Islands',
    'Falkland Islands', 'Gibraltar', 'Montserrat', 'Pitcairn',
    'Saint Helena', 'Turks and Caicos Islands',
    # Dépendances de la Couronne
    'Guernsey', 'Isle of Man', 'Jersey',
    # Territoires FR
    'French Guiana', 'French Polynesia', 'Guadeloupe', 'Martinique',
    'Mayotte', 'New Caledonia', 'Reunion', 'Saint Barthelemy',
    'Saint Martin (French part)', 'Saint Pierre and Miquelon', 'Wallis and Futuna',
    # Territoires NL
    'Aruba', 'Bonaire Sint Eustatius and Saba', 'Curacao', 'Sint Maarten (Dutch part)',
    # Autres territoires
    'Cook Islands', 'Faroe Islands', 'Greenland', 'Kosovo', 'Niue',
    'Palestine', 'Tokelau',
]

mask = df['country'].isin(regions)
df_region = df[mask].copy()
df_country = df[~mask].copy()

print(f"df_country: {df_country.shape} ({df_country['country'].nunique()} pays)")
print(f"df_region: {df_region.shape} ({df_region['country'].nunique()} régions/territoires)")

df_country: (432706, 40) (193 pays)
df_region: (125552, 40) (56 régions/territoires)


In [21]:
df_country.country.unique().size

193

In [24]:
df_country.groupby('country')['new_cases_per_million'].sum()

country
Afghanistan                           5796.468293
Albania                             119264.510644
Algeria                               5990.559133
Andorra                             602280.424475
Angola                                3016.329888
Antigua and Barbuda                  98158.319871
Argentina                           222845.630576
Armenia                             157901.058010
Australia                           452699.014736
Austria                             671151.142931
Azerbaijan                           81251.583853
Bahamas                              98422.305065
Bahrain                             454275.951117
Bangladesh                           12116.045226
Barbados                            386580.666185
Belarus                             108367.926709
Belgium                             421508.022719
Belize                              177635.254419
Benin                                 2037.573003
Bhutan                               80921

In [4]:
# Total des cas par million à la dernière date pour chaque pays
last_date = df_country.groupby('country')['date'].max()
df_last = df_country.merge(last_date.rename('last_date'), on='country')
df_last = df_last[df_last['date'] == df_last['last_date']]

fig = px.histogram(df_last, x='total_cases_per_million', nbins=30,
                   labels={'total_cases_per_million': 'Total cases per million'},
                   title='Distribution du total de cas par million (dernière date)')
fig.update_layout(yaxis_title='Nombre de pays')
fig.show()

In [5]:
df_france = df_country[df_country['country'] == 'France'].copy()
df_france['date'] = pd.to_datetime(df_france['date'])
df_france = df_france.set_index('date').sort_index()
df_france_week = df_france[['new_cases_per_million']].resample('W').sum()

fig = px.line(df_france_week, y='new_cases_per_million',
              labels={'date': 'Date', 'new_cases_per_million': 'Nouveaux cas par million (par semaine)'},
              title='Évolution du COVID-19 en France (nouveaux cas par million, pas hebdomadaire)')
fig.show()

In [6]:
df_brazil = df_country[df_country['country'] == 'Brazil'].copy()
df_brazil['date'] = pd.to_datetime(df_brazil['date'])
df_brazil = df_brazil.set_index('date').sort_index()
df_brazil_week = df_brazil[['new_cases_per_million']].resample('W').sum()

fig = px.line(df_brazil_week, y='new_cases_per_million',
              labels={'date': 'Date', 'new_cases_per_million': 'Nouveaux cas par million (par semaine)'},
              title='Évolution du COVID-19 au Brésil (nouveaux cas par million, pas hebdomadaire)')
fig.show()